# AI Music OS — Kokoro TTS V4

Stable Google Colab version for Kokoro TTS.

This notebook is designed to avoid unnecessary upgrades of Colab's PyTorch, NumPy and SciPy stack.

**Python 3.13:** Kokoro and Misaki are installed from their upstream GitHub repositories.

**Important:** Do not run random `pip install -U numpy`, `scipy`, `torch` or `setuptools` commands after this setup.

In [ ]:
# STEP 1 — Google Drive + basic environment

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import subprocess
        
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / 'kokoro'

CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE / 'transformers')

print('Python:', sys.version)
print('Python executable:', sys.executable)
print('Drive root:', ROOT)
print('HuggingFace cache:', CACHE)
print('Output folder:', OUT)

# Install only the required system package.
subprocess.run(
    ['apt-get', '-qq', '-y', 'install', 'espeak-ng'],
    check=True
)

print('espeak-ng: OK')

In [ ]:
# STEP 2 — Check Colab's core packages
# IMPORTANT: We do NOT blindly upgrade NumPy/SciPy/Torch.

import sys
        
def get_version(package):
    try:
        module = __import__(package)
        return getattr(module, '__version__', 'unknown')
    except Exception as e:
        return 'ERROR: ' + str(e)

print('NumPy :', get_version('numpy'))
print('SciPy :', get_version('scipy'))
print('Torch :', get_version('torch'))
print('Python:', sys.version.split()[0])

# The fresh Colab runtime normally already has a compatible stack.
# We intentionally do not upgrade these packages here.

In [ ]:
# STEP 3 — Install compatible Kokoro + Misaki

import subprocess
import sys
        
def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir']
    cmd.extend(packages)
    print('Installing:', ' '.join(packages))
    subprocess.run(cmd, check=True)

# Misaki from upstream GitHub.
# This avoids the old PyPI Python-version restriction on Python 3.13.
pip_install('git+https://github.com/hexgrad/misaki.git')

# Kokoro from upstream GitHub.
# --no-deps prevents pip from replacing Colab's NumPy/Torch stack.
pip_install('--no-deps', 'git+https://github.com/hexgrad/kokoro.git')

# Install only runtime packages that Kokoro needs and that are safe
# to install without replacing NumPy/Torch.
pip_install(
    'loguru',
    'huggingface_hub',
    'soundfile'
)

# Transformers is already normally present in Colab.
# Install it only if importing it fails later; do not force an upgrade here.

print('\nKokoro + Misaki installation completed.')

In [ ]:
# STEP 4 — Verify NumPy/SciPy/Transformers before Kokoro

import numpy as np
import scipy
import torch

print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: Not available — Kokoro will use CPU')

print('\nTesting Transformers AlbertModel import...')

try:
    from transformers import AlbertModel
    print('AlbertModel: OK')
except Exception as e:
    print('\nERROR: Transformers/NumPy/SciPy environment is broken.')
    print('Details:', repr(e))
    raise

In [ ]:
# STEP 5 — Import Kokoro and create pipeline

import kokoro
from kokoro import KPipeline

print('Kokoro module:', kokoro.__file__)
print('Kokoro version:', getattr(kokoro, '__version__', 'unknown'))

print('\nCreating Kokoro pipeline...')

pipeline = KPipeline(lang_code='a')

print('Kokoro pipeline: OK')

In [ ]:
# STEP 6 — Kokoro voices + audio generation

import numpy as np
import soundfile as sf
from pathlib import Path
import time
        
VOICE_CHOICES = [
    'af_heart',
    'af_bella',
    'af_nicole',
    'af_sarah',
    'af_sky',
    'am_adam',
    'am_michael',
    'bf_emma',
    'bf_isabella',
    'bm_george',
    'bm_lewis'
]

SAMPLE_RATE = 24000

def generate_audio(text, voice='af_heart'):
    text = (text or '').strip()

    if not text:
        raise ValueError('Text is empty.')

    if voice not in VOICE_CHOICES:
        raise ValueError(
            f'Invalid voice: {voice}. Available voices: {VOICE_CHOICES}'
        )

    chunks = []

    print('Generating audio...')
    print('Voice:', voice)

    # Current Kokoro API returns:
    # (graphemes, phonemes, audio)
    for item in pipeline(text, voice=voice):
        if isinstance(item, tuple) and len(item) >= 3:
            audio = item[2]
        elif isinstance(item, dict):
            audio = item.get('audio')
        else:
            audio = getattr(item, 'audio', None)
        
        if audio is None:
            raise RuntimeError(
                f'Unsupported Kokoro output type: {type(item)}'
            )
        
        audio = np.asarray(audio)
        
        if audio.size == 0:
            continue
        
        chunks.append(audio)
        
    if not chunks:
        raise RuntimeError('Kokoro returned no audio.')
        
    audio = np.concatenate(chunks)
        
    filename = f'kokoro_{int(time.time())}.wav'
    output_path = OUT / filename
        
    sf.write(str(output_path), audio, SAMPLE_RATE)
        
    print('Audio saved:', output_path)
        
    return str(output_path)
        
print('Available voices:')
for v in VOICE_CHOICES:
    print(' -', v)

In [ ]:
# STEP 7 — Automatic smoke test

TEST_TEXT = (
    'Hello. This is a test of the AI Music OS Kokoro voice generator.'
)

test_file = generate_audio(
    TEST_TEXT,
    voice='af_heart'
)

print('\n========================================')
print('KOKORO TEST PASSED')
print('Output:', test_file)
print('========================================')

In [ ]:
# STEP 8 — Gradio UI

import gradio as gr

def ui_generate(text, voice):
    try:
        return generate_audio(text, voice)
    except Exception as e:
        raise gr.Error(str(e))
        
demo = gr.Interface(
    fn=ui_generate,
    inputs=[
        gr.Textbox(
            lines=8,
            label='Text',
            placeholder='Enter text here...',
            value=(
                'Hello. This is the AI Music OS Kokoro voice generator.'
            )
        ),
        gr.Dropdown(
            choices=VOICE_CHOICES,
            value='af_heart',
            label='Voice'
        )
    ],
    outputs=gr.Audio(
        label='Generated Audio',
        type='filepath'
    ),
    title='AI Music OS — Kokoro TTS V4',
    description=(
        'Generate high-quality speech using Kokoro TTS. '
        'Generated files are saved to Google Drive.'
    )
)

print('Launching Gradio...')
demo.launch(share=True, debug=True)